In [6]:
import pandas as pd
import nltk
nltk.download('stopwords')
from nltk.stem.porter import PorterStemmer 
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.ensemble import RandomForestClassifier
import pickle
import dill

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\BTQS5984\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [2]:
data = pd.read_csv("../Data/GeneratedData/NewsClasses(Other=0, Buissness=1).csv")

In [3]:
data.head()

,News,Class
0,china census shows population aging rapidly ch...,0
1,5 things to know before the stock market opens...,1
2,duke adjusts to life with irving kyrie irving'...,0
3,back to the future for at&t at&t is once again...,1
4,"we should have been safe with sony, say gamers...",1


In [4]:
news = data.News.to_list()
clas = data.Class.to_list()

In [7]:
corpus = []
for inst in news:
    inst = inst.lower()
    inst = inst.split()
    ps = PorterStemmer()
    inst = [ps.stem(word) for word in inst if not word in set(stopwords.words('english'))]
    inst = ' '.join(inst)
    corpus.append(inst)

In [9]:
corpus

["china censu show popul age rapidli china' popul age rapidly, govern said thursday, though leader refus relax strict famili plan control ...",
 '5 thing know stock market open friday (june 19) stock futur ralli report china plan acceler purchas u.s. agricultur products.',
 "duke adjust life irv kyri irving' return meant shuffl duke coach mike krzyzewski, consid good problem, hindranc",
 'back futur at&t at&t incit worri competit consum suffer deal buy t-mobil $39 billion.',
 'safe sony, say gamer soni playstat gamer express shock disappoint wednesday massiv data hack names, address credit-card detail might stolen playstat network.',
 "good news, bad news, pc maker employe employ realiz need option remot work, buy them, they'r go buy again.",
 "libya say regret attack foreign embassi libya' govern said sunday regret attack sever european embassies, say polic forc overpow crowd anger nato airstrik kill son muammar gaddafi.",
 "hidden workshop add libyan rebels' arsen clandestin network 

In [8]:
def text_transformer(txt):
    from nltk.stem.porter import PorterStemmer 
    from nltk.corpus import stopwords
    if isinstance(txt, str):
        txt = txt.lower()
        txt = txt.split()
        ps = PorterStemmer()
        txt = [ps.stem(word) for word in txt if not word in set(stopwords.words('english'))]
        txt = ' '.join(txt)
        return [txt]
    else:
        print("Enter a valide format (str) !")

In [13]:
vectorizer = CountVectorizer()
matrix_of_words = vectorizer.fit_transform(corpus).toarray()
selector = SelectKBest(chi2, k=900)

In [15]:
X = selector.fit_transform(matrix_of_words, clas)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, clas, test_size = 0.20, random_state = 0)

In [18]:
classifier = RandomForestClassifier(n_estimators=100, random_state=111)
classifier.fit(X_train, y_train)

RandomForestClassifier(random_state=111)

In [19]:
y_pred = classifier.predict(X_test)

In [20]:
cv = confusion_matrix(y_test, y_pred)
cv

array([[176,  35],
       [ 17, 196]], dtype=int64)

In [22]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.8773584905660378

In [23]:
save_transformer = open("../PythonObjects/NewsClassifierObjects/Transformer.pickle","wb")
dill.dump(text_transformer, save_transformer)
save_transformer.close()

In [24]:
save_vectorizer = open("../PythonObjects/NewsClassifierObjects/Vectorizer.pickle","wb")
pickle.dump(vectorizer, save_vectorizer)
save_vectorizer.close()

In [25]:
save_selector = open("../PythonObjects/NewsClassifierObjects/Selector.pickle","wb")
pickle.dump(selector, save_selector)
save_selector.close()

In [26]:
save_classifier = open("../PythonObjects/NewsClassifierObjects/Classifier(RF).pickle","wb")
pickle.dump(classifier, save_classifier)
save_classifier.close()

In [27]:
t = dill.load(open("../PythonObjects/NewsClassifierObjects/Transformer.pickle", "rb"))
v = pickle.load(open("../PythonObjects/NewsClassifierObjects/Vectorizer.pickle", "rb"))
s = pickle.load(open("../PythonObjects/NewsClassifierObjects/Selector.pickle", "rb"))
c = pickle.load(open("../PythonObjects/NewsClassifierObjects/Classifier(RF).pickle", "rb"))

In [28]:
text = t("The solution to this problem is because input is just a String, but what is needed is a list (or an iterable) containing a single element (which is nothing but the String itself).")
vector = v.transform(text)
features = s.transform(vector)
prediction = c.predict(features)
prediction

array([0])